In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc samplomatic
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Executor-Schnellstart

<Accordion>
<AccordionItem title="Paketversionen">

Der Code auf dieser Seite wurde mit den folgenden Anforderungen entwickelt.
Wir empfehlen, diese Versionen oder neuere zu verwenden.

```
qiskit[all]~=2.4.0
qiskit-ibm-runtime~=0.46.1
samplomatic~=0.18.0
```
</AccordionItem>
</Accordion>

Ähnlich wie das [Sampler](/guides/get-started-with-sampler)-Primitive sampelt Executor Ausgaberegister aus Quantenschaltungs-Ausführungen, hat jedoch keine eingebaute Fehlerunterdrückung oder -minderung. Stattdessen ist er Teil des [directed-execution-Modells](/guides/directed-execution-model), das die Bausteine bereitstellt, um Design-Absichten auf der Client-Seite zu erfassen, und die aufwendige Generierung von Schaltungsvarianten auf die Server-Seite verlagert. Executor folgt den Direktiven aus Circuit-Annotationen und Optionen, generiert und bindet Parameterwerte, führt die gebundenen Schaltungen auf der Hardware aus und gibt die Ausführungsergebnisse und Metadaten zurück. Er trifft keine impliziten Entscheidungen für dich und gibt dir volle Kontrolle und Transparenz.

> **Note:** Das Qiskit-Paket hat noch keine Basisklasse für das Executor-Primitive.

## Bevor du beginnst
Einige der Codebeispiele auf dieser Seite verwenden `samplex`, das Teil des Samplomatic-Pakets ist. Deshalb musst du Samplomatic installieren, bevor du diese Codeblöcke ausführst, wie im folgenden Codeblock gezeigt. Weitere Informationen findest du in der [Samplomatic-Dokumentation](https://qiskit.github.io/samplomatic).

In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService, Executor
from qiskit_ibm_runtime.quantum_program import QuantumProgram
from qiskit.circuit import QuantumCircuit
from qiskit.transpiler import generate_preset_pass_manager
from samplomatic.transpiler import generate_boxing_pass_manager
from samplomatic import build

# Initialize the service and choose a backend
service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

In [2]:
print(backend)

<IBMBackend('ibm_fez')>


### 2. Create and transpile a circuit

You need at least one circuit to use the Executor primitive.  It can optionally have parameters.

In [3]:
# Generate the circuit
circuit = QuantumCircuit(2)
circuit.h(0)
circuit.h(1)
circuit.cz(0, 1)
circuit.h(1)

# Using `measure_all` automatically creates the necessary
# classical registers.
circuit.measure_all()

The circuit needs to be transformed to only use instructions supported by the QPU (referred to as *instruction set architecture (ISA)* circuits). Use the transpiler to do this.

In [4]:
# Transpile the circuit
preset_pass_manager = generate_preset_pass_manager(
    backend=backend, optimization_level=0
)
isa_circuit = preset_pass_manager.run(circuit)

### 2. Einen Circuit erstellen und transpilieren
Du benötigst mindestens einen Circuit, um das Executor-Primitive zu verwenden. Er kann optional Parameter haben.

In [5]:
# Initialize an empty program
program = QuantumProgram(shots=25)

# Append the circuit to the program
program.append_circuit_item(isa_circuit)

Der Circuit muss so transformiert werden, dass er nur Anweisungen verwendet, die vom QPU unterstützt werden (sogenannte *Instruction Set Architecture (ISA)*-Circuits). Verwende dafür den Transpiler.

In [6]:
# Generate a boxing pass manager to group gates
# and measurements into boxes and add
# a`Twirl` annotation.
boxes_pm = generate_boxing_pass_manager(
    # Add gate twirling
    enable_gates=True,
    # Add measurement twirling
    enable_measures=True,
)

boxed_circuit = boxes_pm.run(isa_circuit)
boxed_circuit.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/get-started-with-executor/extracted-outputs/245a4574-3ce9-4f77-98c8-af32cde8ac01-0.svg" alt="Output of the previous code cell" />

### 3. Ein `QuantumProgram` initialisieren
Initialisiere ein `QuantumProgram` mit deiner Arbeitslast. Ein `QuantumProgram` besteht aus `QuantumProgramItems`. In der Regel enthält jedes Element einen Circuit, einen Satz von Parameterwerten und möglicherweise ein `samplex`, um den Circuit-Inhalt zu randomisieren. Alle Details findest du unter [Executor-Eingaben und -Ausgaben](/guides/executor-input-output).

Die folgende Zelle initialisiert ein `QuantumProgram` und gibt an, 25 Shots durchzuführen. Anschließend wird der transpilierte Ziel-Circuit angehängt.

In [7]:
# Build the template circuit and the samplex
template_circuit, samplex = build(boxed_circuit)

# Append the template circuit and samplex as a `samplex_item`
program.append_samplex_item(
    template_circuit,
    samplex=samplex,
    shape=(num_randomizations := 20,),
)

### 4. Optional: Gates und Messungen in annotierte Boxen gruppieren
Das Gruppieren von Anweisungen in Boxen und deren Annotierung ist der primäre Weg, um deine Absicht anzugeben. Im folgenden Beispiel verwenden wir `generate_boxing_pass_manager` und seine Twirling-Parameter, um Zwei-Qubit-Gates und Messungen in Boxen zu gruppieren und Twirling-Annotierung anzuwenden.

In [8]:
# Initialize an Executor with the default options
executor = Executor(mode=backend)

# Submit the job
job = executor.run(program)
job

<RuntimeJobV2('d8286580bvlc73d1vmsg', 'executor')>

In [9]:
# Retrieve the result
result = job.result()

### 6. Executor aufrufen und Ergebnisse abrufen
Führe das `QuantumProgram` auf einem IBM&reg;-Backend aus, indem du das `Executor`-Primitive mit Standardoptionen verwendest. Unter [Executor-Optionen](/guides/executor-options) erfährst du mehr über die verfügbaren Optionen.